In [55]:
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.documents import Document

import os
from dotenv import load_dotenv

load_dotenv()

# Interfaz de LangChain para usar la función de embedding de Gemini
GEMINI_API_KEY = os.getenv('GEMINI_API_KEY')
embeddings = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001", google_api_key=GEMINI_API_KEY)

# Ruta en la que se guardarán las colecciones de Chroma
PERSIST_DIRECTORY="./chroma_db"

# Clientes de ChromaDB usando la interfaz de LangChain
# uno por cada colección (movies, people y reviews)

# vector_store_movies = Chroma(collection_name='movies', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)
# vector_store_people = Chroma(collection_name='people', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)
# vector_store_reviews = Chroma(collection_name='reviews', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

In [56]:
import json
import requests

# Para obtener información de una película
# usamos la API de TMDB
TMDB_API_KEY = os.getenv('TMDB_API_KEY')
TMDB_HEADERS = {
      "accept": "application/json",
      "Authorization": f"Bearer {TMDB_API_KEY}"
}

def get_movies_info(title):
  url = f"https://api.themoviedb.org/3/search/movie?query={title}&include_adult=false&language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_reviews(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/reviews?language=en-US&page=1"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_movie_cast(movie_id):
  url = f"https://api.themoviedb.org/3/movie/{movie_id}/credits?language=en-US"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)

def get_person_details(person_id):
  url = f"https://api.themoviedb.org/3/person/{person_id}"

  response = requests.get(url, headers=TMDB_HEADERS)
  return json.loads(response.text)


## Añadir películas a la colección "movies" de ChromaDB

In [57]:
def add_movies_to_collection(title: str):
  """
  Busca infomación de una película a partir de su título.

  Args:
    title (str): El título de la película sobre la que buscamos información.
  """
  movies_info = get_movies_info(title)

  # Vector Store de la base de datos de ChromaDB
  vector_store_movies = Chroma(collection_name='movies', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

  print("Added these movies to collection ('movies'):")

  # Documentos que se añadirán a la base de datos
  # Los documentos en LangChain tienen:
  # - pageContent (str): El texto con el contenido.
  # - metadata (dict): Metadatos, igual que en Chroma.
  # - id (str): Identificador único por documento.
  docsToAdd : list[Document] = []

  def get_cast_info(movie):
    cast_found = get_movie_cast(str(movie['id']))
    director = ""
    cast = ""

    for person in cast_found['cast']:
      if person['known_for_department'] == 'Directing':
        director += person['name'] + ", "
      else:
        cast += person['name'] + ", "
    return director, cast
  
  def get_metadata(movie, director, cast):
    return {
        "movie_title" : movie['title'],
        "director"    : director,
        "cast"        : cast,
        "popularity"  : movie['popularity'],
        "release_date": movie['release_date'],
        "vote_average": movie['vote_average'],
        "vote_count"  : movie['vote_count']
    }
  
  for movie in movies_info['results']:
    # Consultamos con nuestra colección
    result = vector_store_movies.get(
      ids=[str(movie['id'])],
    )

    # Si no existe, la añade
    if not result['ids'] and movie['overview']:

      # Primero obtenemos al cast
      director, cast = get_cast_info(movie)

      # Añadimos el documento con la información de la peli
      docsToAdd.append(Document(
        page_content=movie['overview'],
        metadata=get_metadata(movie, director, cast),
        id=str(movie['id'])
      ))

      print(docsToAdd)

      print(f"Added {movie['title']}.")

  if len(docsToAdd) > 0:
    vector_store_movies.add_documents(docsToAdd)

  print(f"Added {len(docsToAdd)} movies.")

# Ejemplo de cómo usarlo junto a la búsqueda en TMDB
add_movies_to_collection("todo sobre mi madre")

Added these movies to collection ('movies'):
[Document(id='99', metadata={'movie_title': 'All About My Mother', 'director': '', 'cast': 'Cecilia Roth, Marisa Paredes, Candela Peña, Antonia San Juan, Penélope Cruz, Rosa María Sardà, Fernando Fernán Gómez, Fernando Guillén, Toni Cantó, Eloy Azorín, Carlos Lozano, Manuel Morón, José Luis Torrijo, Juan José Otegui, Carmen Balagué, Malena Gutiérrez, Yael Barnatán, Carme Fortuny, Patxi Freytez, Juan Marquez, Michel Ruben, Daniel Lanchas, Rosa Manaut, Carlos García Cambero, Agustín Almodóvar, Paz Sufrategui, Lola García, Esther García, Inma Subirà, Cayetana Guillén Cuervo, Alexia Pardo, Lluís Pasqual, Fito Páez, ', 'popularity': 4.8729, 'release_date': '1999-04-16', 'vote_average': 7.629, 'vote_count': 1985}, page_content='Following the tragic death of her teenage son, Manuela travels from Madrid to Barcelona in an attempt to contact the long-estranged father the boy never knew. She reunites with an old friend, an outspoken transgender sex wo

## Añadir reviews a la colección "reviews" de ChromaDB

In [58]:
def add_reviews_to_collection(title: str):
  """
  Busca reviews y opiniones sobre una película en concreto.

  Args:
    title (str): El título de la película sobre la que necesitamos opiniones.
  """

  print(f"Added these reviews for the movie '{title}' to collection ('reviews'):")

  movies = get_movies_info(title)
  for movie in movies['results']:
    add_reviews_from_id(movie['id'], movie['title'])
    

def add_reviews_from_id(movie_id, movie_title):  
  vector_store_reviews = Chroma(collection_name='reviews', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

  # Documentos que se añadirán a la base de datos
  # Los documentos en LangChain tienen:
  # - pageContent (str): El texto con el contenido.
  # - metadata (dict): Metadatos, igual que en Chroma.
  # - id (str): Identificador único por documento.
  docsToAdd : list[Document] = []

  # Consultamos con nuestra colección a través
  # del id de la película
  result = vector_store_reviews.get(
    where={"movie_id": movie_id},
  )

  # Si NO hay reviews para esa película,
  # se intentan añadir
  if (len(result['ids']) == 0):
    reviews = get_movie_reviews(movie_id)

    def get_metadata(review):
      if review["author_details"]["rating"]:
        return {
            "movie_title": movie_title,
            "movie_id": movie_id,
            "author"  : review["author"],
            "rating"  : review["author_details"]["rating"]
        }
      else:
        return {
            "movie_title": movie_title,
            "movie_id": movie_id,
            "author"  : review["author"],
        }

    for review in reviews['results']:

      if review['content']:
        docsToAdd.append(Document(
          page_content=review['content'],
          metadata=get_metadata(review),
          id=str(review['id'])
        ))

    if len(docsToAdd) > 0:
      vector_store_reviews.add_documents(docsToAdd)

    print(f"Reviews for '{movie_title}': Added {len(docsToAdd)} reviews.")
  else:
    print(f"Reviews for '{movie_title}' already in database.")

add_reviews_to_collection('the elephant man')


Added these reviews for the movie 'the elephant man' to collection ('reviews'):
Reviews for 'The Elephant Man': Added 2 reviews.
Reviews for 'The Elephant Man': Added 0 reviews.
Reviews for 'The Devildom Elephant Man': Added 0 reviews.
Reviews for 'The Elephant Man': Added 0 reviews.
Reviews for 'The Terrible Elephant Man Revealed': Added 0 reviews.
Reviews for 'Revenge of the Elephant Man': Added 0 reviews.
Reviews for 'The Curse of the Elephant Man': Added 0 reviews.
Reviews for 'The Elephant Man’s Sound, Tracked.': Added 0 reviews.
Reviews for 'Joseph Merrick: The Real Elephant Man': Added 0 reviews.
Reviews for 'The Man With Elephant Hands': Added 0 reviews.


## Añadir actores a la colección "people" de ChromaDB

In [59]:
import json
import requests

def add_person_to_collection(name: str):
  """
  Busca información de una persona del cast de una película.

  Args:
    name (str): El nombre de la persona que el usuario está buscando.
  """
  # Buscamos los el nombre para encontrar ilos d
  url = f"https://api.themoviedb.org/3/search/person?query={name}&include_adult=false&language=en-US&page=1"
  r = requests.get(url, headers=TMDB_HEADERS)
  response = json.loads(r.text)

  print("Added these people to collection ('people'): ")

  for person in response['results']:
    add_person_from_id(person['id'])

def add_person_from_id(person_id):
  vector_store_people = Chroma(collection_name='people', embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

  # Documentos que se añadirán a la base de datos
  # Los documentos en LangChain tienen:
  # - pageContent (str): El texto con el contenido.
  # - metadata (dict): Metadatos, igual que en Chroma.
  # - id (str): Identificador único por documento.
  docsToAdd : list[Document] = []

  details = get_person_details(person_id)

  # Consultamos con nuestra colección
  # por si ya hemos guardado a esa persona
  result = vector_store_people.get(
    ids=[str(details['id'])]
  )

  if not result['ids'] and details['biography']:
    
    # Añadir género de la persona
    gender = "Not set"
    if details['gender'] == 1:
      gender = "female"
    elif details['gender'] == 2:
      gender = "male"
    elif details['gender'] == 3:
      gender = "non binary"

    # add_documents pide una lista, así que creamos una
    # con la persona que vamos a añadir
    docsToAdd.append(Document(
      page_content=details['biography'],
      metadata={'name': details['name'], 'department': details['known_for_department'], 'gender': gender},
      id=str(details['id'])
    ))

    vector_store_people.add_documents(docsToAdd)

    print(f"Added {details['name']}")
  else:
    print(f"{details['name']} already exists in collection")

add_person_to_collection("penelope cruz")

Added these people to collection ('people'): 
Penélope Cruz already exists in collection
Aliyah Penelope Cruz already exists in collection


In [63]:
def query_col(query: str, collection: str):
    """
    Consulta a una colección de las disponibles (movies, reviews y people)
    con la pregunta que ha hecho el usuario, para responder con información veraz.

    Args:
        query (str): La pregunta del usuario.
        collection (str): La colección a consultar: 'movies', 'reviews' o 'people'.
    """

    # Cogemos la colección dependiendo de lo que necesitemos
    vector_store = Chroma(collection_name=collection, embedding_function=embeddings, persist_directory=PERSIST_DIRECTORY)

    # Similarity search hace una búsqueda utilizando la función de embeddings del vector_store
    results = vector_store.similarity_search(
        query=query,
        k=10,
    )

    # Lo que se devolverá, un array de diccionarios que tendrán dos propiedades:
    # - page_content (str): Los documentos en Chroma, el texto.
    # - metadata (dict): Metadata de los resultados (título, votos...).
    formatted_results: list[dict] = []

    for doc in results:
        formatted_results.append({"page_content": doc.page_content, "metadata": doc.metadata})

    return formatted_results

query_col('love drama in france with dogs','movies')

[{'page_content': 'Two young gay soccer players get caught up between the politics of the game and the politics of love.',
  'metadata': {'fecha_estreno': '2018-02-22',
   'cast': 'Max Hubacher, Aaron Altaras, Jessy Moravec, Jürg Plüss, Doro Müggler, Andreas Matti, Joris Gratwohl, Scherwin Amini, Fabrizio Borsani, Julian Koechlin, Beat Marti, Matthias Neukirch, Elias Reichert, Gabriel Noah Maurer, Annina Polivka, Marin Blülle, Niklas Löffler, Mats Kampen, Mersiha Husagić, Tabita Johannes, Anna-Katharina Müller, Nina Fischer, Stallone Anderson, Manuel Pereira, Tom Burri, Nina Mariel Kohler, Christian Liniger, Joshua Schmidli, Raphael Spicher, Sirak Gebrehiwet, Maruan Samson, Hanibal Samson, Francesco Molinaro, Nathan Francisco, Tim Bohren, Mohammad Khodabakshi, Valentino Gigante, Samuel Oluwafemi Ojo, Jakub Hlobil, Filip Lovrinovic, Kamil Piekarczyk, Uche Santschi, Dario Rumenovic, Jana Pensa',
   'voto_promedio': 7.142,
   'voto_count': 144,
   'popularidad': 5.2605,
   'director': 'Ma